# Enterprise AI Chatbot Pipeline Walkthrough

This notebook demonstrates every major stage: user query, intent detection, planning, tool selection, execution, retrieval, reranking placeholder, prompt construction, generation, guardrails, judging, metrics, logging, and final response.

In [1]:
from app.dependencies import get_orchestrator
from models.schemas import ChatRequest
from config.settings import Settings
from rag.pipeline import RagPipeline

orchestrator = get_orchestrator()

## 1. User Query

In [2]:
query = "Retrieve the enterprise data policy and summarize access requirements"
request = ChatRequest(query=query, session_id="notebook-demo")
request

ChatRequest(query='Retrieve the enterprise data policy and summarize access requirements', session_id='notebook-demo', files=[], metadata={})

## 2-4. Intent Detection, Planner Output, Tool Selection

In [3]:
plan = await orchestrator.planner.plan(request.query, request.files)
plan.model_dump()

{'intent': 'answer_user_query',
 'can_answer_directly': False,
 'steps': [{'tool': <ToolName.rag: 'rag'>,
   'reason': 'Planner selected rag',
   'args': {}}],
 'risks': []}

## 5-8. Tool Execution, Retrieved Documents, Hybrid Search, Reranking

In [4]:
orchestrator.rag.ingest_path(__import__('pathlib').Path('../sample_data/policies.md'))
rag_result = await orchestrator.rag.answer(request.query)
rag_result.model_dump()

{'tool': <ToolName.rag: 'rag'>,
 'success': True,
 'output': {'contexts': [{'id': '..\\sample_data\\policies.md:section:0',
    'score': 2.1288360878805945,
    'text': '# Enterprise Data Policy\n\nCustomer records may be used only for approved support, analytics, and compliance workflows.\n'},
   {'id': '..\\sample_data\\policies.md:section:2',
    'score': 0.725586174531274,
    'text': '## Access\n\nSQL access must be read-only for AI assistant workloads unless a separate human approval workflow is configured.\n'},
   {'id': '..\\sample_data\\policies.md:section:1',
    'score': 0.49491755935805337,
    'text': '## Retention\n\nOperational logs are retained for 30 days by default. Audit logs are retained according to the compliance schedule.\n'}]},
 'latency_ms': 5.468500079587102,
 'citations': [{'source_id': '..\\sample_data\\policies.md:section:0',
   'title': '..\\sample_data\\policies.md',
   'location': None,
   'score': 2.1288360878805945,
   'metadata': {}},
  {'source_id': 

## 9-15. Prompt, Generation, Guardrails, Judge, Metrics, Logging, Final Response

In [5]:
response = await orchestrator.chat(request)
{
    'answer': response.answer,
    'plan': response.plan.model_dump(),
    'timeline': response.timeline,
    'judge': response.judge.model_dump(),
    'metrics': response.metrics.model_dump(),
    'citations': [c.model_dump() for c in response.citations],
}

{'answer': 'I analyzed the available context and produced a grounded response from the executed tools.',
 'plan': {'intent': 'answer_user_query',
  'can_answer_directly': False,
  'steps': [{'tool': <ToolName.rag: 'rag'>,
    'reason': 'Planner selected rag',
    'args': {}}],
  'risks': []},
 'timeline': [{'agent': 'guardrail', 'allowed': True, 'reasons': []},
  {'agent': 'planner',
   'plan': {'intent': 'answer_user_query',
    'can_answer_directly': False,
    'steps': [{'tool': <ToolName.rag: 'rag'>,
      'reason': 'Planner selected rag',
      'args': {}}],
    'risks': []}},
  {'agent': 'rag_agent', 'success': True, 'latency_ms': 0.367600005120039},
  {'agent': 'validation', 'issues': []},
  {'agent': 'judge',
   'scores': {'correctness': 0.8,
    'groundedness': 0.8,
    'faithfulness': 0.8,
    'completeness': 0.7,
    'relevance': 0.8,
    'hallucination_score': 0.19999999999999996,
    'notes': 'Heuristic judge active. Configure an LLM judge provider for deeper scoring.'}}],